# Image Captioning using CLIP + GPT-2 (Multimodal)
This notebook demonstrates a simple multimodal pipeline:
Image → CLIP encoder → GPT‑2 caption generation

In [ ]:
!pip install transformers torch torchvision pillow matplotlib

In [ ]:
import torch
from PIL import Image
import matplotlib.pyplot as plt

from transformers import CLIPProcessor, CLIPModel
from transformers import GPT2Tokenizer, GPT2LMHeadModel

In [ ]:
# Load CLIP model
clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
clip_model.to(device)

In [ ]:
# Load GPT2 caption generator
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt_model = GPT2LMHeadModel.from_pretrained('gpt2')

gpt_model.to(device)
gpt_model.eval()

In [ ]:
# Load image
image_path = 'sample.jpg'
image = Image.open(image_path).convert('RGB')

plt.imshow(image)
plt.axis('off')

In [ ]:
# Extract image features using CLIP
inputs = clip_processor(images=image, return_tensors='pt').to(device)

with torch.no_grad():
    image_features = clip_model.get_image_features(**inputs)

image_features = image_features / image_features.norm(dim=-1, keepdim=True)
print('Feature shape:', image_features.shape)

In [ ]:
# Caption prompt
prompt = 'A photo of'
inputs = tokenizer(prompt, return_tensors='pt').to(device)

In [ ]:
# Generate caption
output = gpt_model.generate(
    inputs.input_ids,
    max_length=30,
    num_beams=5,
    early_stopping=True
)

caption = tokenizer.decode(output[0], skip_special_tokens=True)
print('Generated Caption:', caption)

In [ ]:
# Function for caption generation
def generate_caption(image_path):
    image = Image.open(image_path).convert('RGB')

    inputs = clip_processor(images=image, return_tensors='pt').to(device)

    with torch.no_grad():
        image_features = clip_model.get_image_features(**inputs)

    prompt = 'A photo of'
    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    output = gpt_model.generate(
        inputs.input_ids,
        max_length=30,
        num_beams=5,
        early_stopping=True
    )

    caption = tokenizer.decode(output[0], skip_special_tokens=True)
    return caption

In [ ]:
# Test the system
caption = generate_caption('sample.jpg')
print('Caption:', caption)